# Evaluacion de modelos ALPR en condiciones dificiles

Este notebook compara varios modelos YOLO compatibles con Ultralytics sobre las mismas imagenes de placas y sus variantes: original, borrosa, sucia/contraste bajo, oscura, lejana y con ruido.

1. Coloca imagenes `.jpg`, `.jpeg` o `.png` en `plate-detector/dataset`.
2. Cambia `MODEL_PATHS` por las rutas de tus pesos `.pt` entrenados para detectar placas.
3. Ejecuta todas las celdas. El CSV queda en `plate-detector/resultados/resultados.csv`.

Con imagenes sin etiquetas se comparan detecciones, confianza y tiempo. Para calcular precision, recall o mAP hacen falta etiquetas ground truth en formato YOLO. El OCR es opcional y se desactiva por defecto.

In [2]:
%pip install -q ultralytics opencv-python pandas matplotlib pillow easyocr

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from pathlib import Path
import time
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from ultralytics import YOLO
except ImportError:
    YOLO = None

PROJECT_DIR = Path.cwd()
DETECTOR_DIR = PROJECT_DIR if PROJECT_DIR.name == 'plate-detector' else PROJECT_DIR / 'plate-detector'
DATASET_DIR = DETECTOR_DIR / 'dataset'
OUTPUT_DIR = DETECTOR_DIR / 'resultados'
MODELS_DIR = DETECTOR_DIR / 'models'
ANNOTATED_DIR = OUTPUT_DIR / 'anotadas'
DATASET_DIR.mkdir(parents=True, exist_ok=True)
ANNOTATED_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATHS = {
    'modelo_1': MODELS_DIR / 'modelo_1.pt',
    'modelo_2': MODELS_DIR / 'modelo_2.pt',
}
CONFIDENCE = 0.25
IMAGE_SIZE = 640
USE_OCR = False
results = pd.DataFrame()
captured_frame = None

image_paths = sorted([p for p in DATASET_DIR.rglob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}])
available_models = {name: Path(path) for name, path in MODEL_PATHS.items() if Path(path).exists()}
print(f'Imagenes encontradas: {len(image_paths)}')
print(f'Modelos disponibles: {list(available_models)}')
if not image_paths:
    print(f'Agrega imagenes a {DATASET_DIR.resolve()}')
if not available_models:
    print(f'Coloca tus pesos .pt en {MODELS_DIR.resolve()}')

Imagenes encontradas: 0
Modelos disponibles: []
Agrega imagenes a C:\Users\jmancebo\Desktop\app-escaner-placa\plate-detector\plate-detector\dataset
No hay pesos .pt disponibles. Edita MODEL_PATHS antes de ejecutar la inferencia.


In [13]:
def make_variants(image):
    variants = {'original': image}
    blurred = cv2.GaussianBlur(image, (0, 0), 5)
    variants['borrosa'] = blurred
    low_contrast = cv2.convertScaleAbs(image, alpha=0.55, beta=18)
    noise = np.random.normal(0, 12, image.shape).astype(np.int16)
    variants['sucia_contraste_bajo'] = np.clip(low_contrast.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    variants['oscura'] = cv2.convertScaleAbs(image, alpha=0.45, beta=-20)
    small = cv2.resize(image, (max(32, image.shape[1] // 3), max(32, image.shape[0] // 3)))
    variants['lejana'] = cv2.resize(small, (image.shape[1], image.shape[0]), interpolation=cv2.INTER_LINEAR)
    grain = np.random.normal(0, 25, image.shape).astype(np.int16)
    variants['ruido'] = np.clip(image.astype(np.int16) + grain, 0, 255).astype(np.uint8)
    return variants

def draw_detections(image, boxes, confidences):
    output = image.copy()
    for box, confidence in zip(boxes, confidences):
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(output, (x1, y1), (x2, y2), (0, 220, 80), 2)
        cv2.putText(output, f'{confidence:.2f}', (x1, max(18, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 220, 80), 2)
    return output

In [14]:
models = {}
if YOLO is not None:
    for name, path in available_models.items():
        try:
            models[name] = YOLO(str(path))
            print(f'Cargado: {name}')
        except Exception as error:
            print(f'No se pudo cargar {name}: {error}')
else:
    print('Ultralytics no esta instalado. Ejecuta la celda de instalacion.')

In [18]:
def capture_from_camera(camera_index=0, window_name='Camara ALPR'):
    camera = cv2.VideoCapture(camera_index, cv2.CAP_DSHOW)
    if not camera.isOpened():
        camera.release()
        raise RuntimeError('No se pudo abrir la camara. Revisa permisos, indice de camara o si otra aplicacion la esta usando.')

    captured = None
    print('Camara activa: ESPACIO captura | ESC cancela')
    try:
        while True:
            ok, frame = camera.read()
            if not ok:
                raise RuntimeError('La camara no devolvio una imagen.')
            cv2.imshow(window_name, frame)
            key = cv2.waitKey(1) & 0xFF
            if key == 32:
                captured = frame.copy()
                break
            if key == 27:
                break
    finally:
        camera.release()
        cv2.destroyAllWindows()
    return captured

captured_frame = capture_from_camera()
if captured_frame is not None:
    camera_path = OUTPUT_DIR / 'imagenes.jpg'
    cv2.imwrite(str(camera_path), captured_frame)
    display_image = cv2.cvtColor(captured_frame, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(10, 6))
    plt.imshow(display_image)
    plt.axis('off')
    plt.title('Captura de camara')
    plt.show()
    print(f'Captura guardada en {camera_path}')
else:
    print('Captura cancelada.')

Camara activa: ESPACIO captura | ESC cancela


KeyboardInterrupt: 

In [ ]:
def scan_camera_capture(frame):
    if frame is None:
        print('No hay captura para analizar.')
        return pd.DataFrame()
    if not models:
        print('No hay modelos cargados. Edita MODEL_PATHS y ejecuta la celda de carga.')
        return pd.DataFrame()

    rows = []
    for model_name, model in models.items():
        started = time.perf_counter()
        try:
            result = model.predict(source=frame, conf=CONFIDENCE, imgsz=IMAGE_SIZE, verbose=False)[0]
            elapsed_ms = (time.perf_counter() - started) * 1000
            boxes = result.boxes.xyxy.cpu().numpy() if result.boxes is not None else np.empty((0, 4))
            confidences = result.boxes.conf.cpu().numpy() if result.boxes is not None else np.array([])
            annotated = draw_detections(frame, boxes, confidences)
            output_path = ANNOTATED_DIR / f'camara_{model_name}.jpg'
            cv2.imwrite(str(output_path), annotated)
            rows.append({'modelo': model_name, 'detecciones': len(boxes), 'confianza_maxima': float(confidences.max()) if len(confidences) else 0.0, 'tiempo_ms': round(elapsed_ms, 2), 'imagen_anotada': str(output_path)})
            preview = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
            plt.figure(figsize=(10, 6))
            plt.imshow(preview)
            plt.title(f'{model_name}: {len(boxes)} deteccion(es), confianza maxima {rows[-1]["confianza_maxima"]:.2f}')
            plt.axis('off')
            plt.show()
        except Exception as error:
            print(f'Error con {model_name}: {error}')
    return pd.DataFrame(rows)

camera_results = scan_camera_capture(captured_frame)
if not camera_results.empty:
    display(camera_results)
    camera_results.to_csv(OUTPUT_DIR / 'resultados_camara.csv', index=False, encoding='utf-8-sig')
    print(f'Resultados de camara guardados en {OUTPUT_DIR / "resultados_camara.csv"}')

In [ ]:
def run_evaluation():
    rows = []
    for image_path in image_paths:
        image = cv2.imread(str(image_path))
        if image is None:
            print(f'No se pudo leer: {image_path}')
            continue
        for condition, variant in make_variants(image).items():
            for model_name, model in models.items():
                started = time.perf_counter()
                try:
                    result = model.predict(source=variant, conf=CONFIDENCE, imgsz=IMAGE_SIZE, verbose=False)[0]
                    elapsed_ms = (time.perf_counter() - started) * 1000
                    boxes = result.boxes.xyxy.cpu().numpy() if result.boxes is not None else np.empty((0, 4))
                    confidences = result.boxes.conf.cpu().numpy() if result.boxes is not None else np.array([])
                    annotated = draw_detections(variant, boxes, confidences)
                    relative_name = image_path.relative_to(DATASET_DIR).with_suffix('')
                    output_path = ANNOTATED_DIR / f'{relative_name.name}_{condition}_{model_name}.jpg'
                    cv2.imwrite(str(output_path), annotated)
                    best_index = int(np.argmax(confidences)) if len(confidences) else None
                    best_box = boxes[best_index].tolist() if best_index is not None else None
                    rows.append({'modelo': model_name, 'imagen': str(image_path), 'condicion': condition, 'detecciones': len(boxes), 'confianza_maxima': float(confidences.max()) if len(confidences) else 0.0, 'mejor_caja_xyxy': best_box, 'tiempo_ms': round(elapsed_ms, 2), 'imagen_anotada': str(output_path)})
                except Exception as error:
                    rows.append({'modelo': model_name, 'imagen': str(image_path), 'condicion': condition, 'detecciones': 0, 'confianza_maxima': 0.0, 'mejor_caja_xyxy': None, 'tiempo_ms': None, 'imagen_anotada': None, 'error': str(error)})
    return pd.DataFrame(rows)

results = run_evaluation() if models and image_paths else pd.DataFrame()
if not results.empty:
    results.to_csv(OUTPUT_DIR / 'resultados.csv', index=False, encoding='utf-8-sig')
    display(results.head())
    display(results.groupby(['modelo', 'condicion'], as_index=False).agg(detecciones_promedio=('detecciones', 'mean'), confianza_promedio=('confianza_maxima', 'mean'), tiempo_ms_promedio=('tiempo_ms', 'mean')))
    output_csv = OUTPUT_DIR / 'resultados.csv'
    print(f'Resultados guardados en {output_csv}')
else:
    print('No hay resultados: revisa imagenes, pesos y dependencias.')

SyntaxError: f-string: expecting '=', or '!', or ':', or '}' (2104096541.py, line 32)

In [10]:
if not results.empty:
    summary = results.pivot_table(index='condicion', columns='modelo', values='confianza_maxima', aggfunc='mean')
    summary.plot(kind='bar', figsize=(12, 5), ylim=(0, 1), title='Confianza maxima promedio por condicion')
    plt.ylabel('Confianza')
    plt.tight_layout()
    plt.show()

    sample = results.iloc[0]
    if sample['imagen_anotada']:
        preview = cv2.cvtColor(cv2.imread(sample['imagen_anotada']), cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(10, 6))
        plt.imshow(preview)
        plt.title(f"{sample['modelo']} | {sample['condicion']} | detecciones: {sample['detecciones']}")
        plt.axis('off')
        plt.show()

NameError: name 'results' is not defined

In [11]:
if USE_OCR and not results.empty:
    import easyocr
    reader = easyocr.Reader(['es', 'en'], gpu=False)
    ocr_rows = []
    for _, row in results[results['detecciones'] > 0].iterrows():
        image = cv2.imread(row['imagen'])
        box = row['mejor_caja_xyxy']
        if box:
            x1, y1, x2, y2 = map(int, box)
            crop = image[max(0, y1):y2, max(0, x1):x2]
            text = reader.readtext(crop, detail=0, paragraph=False)
            ocr_rows.append({**row.to_dict(), 'ocr': ' '.join(text)})
    ocr_results = pd.DataFrame(ocr_rows)
    ocr_results.to_csv(OUTPUT_DIR / 'resultados_ocr.csv', index=False, encoding='utf-8-sig')
    display(ocr_results.head())
else:
    print('OCR desactivado. Cambia USE_OCR = True para probar EasyOCR sobre la mejor deteccion.')

OCR desactivado. Cambia USE_OCR = True para probar EasyOCR sobre la mejor deteccion.
